## Крок 1. Підготовка середовища



In [1]:
%pip install -U langchain langchain-openai langchain-chroma chromadb langchain-text-splitters python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv


possible_env_paths = [
    Path(".env"),
    Path("HW_7_RAG") / ".env",
    Path("HW_6_AI_agent") / ".env",
]

env_path = next((path for path in possible_env_paths if path.exists()), None)

if env_path is None:
    print(".env file not found")
else:
    load_dotenv(env_path)
    print(f"Loaded .env from: {env_path}")

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("API key loaded successfully")
else:
    print("API key not found")

Loaded .env from: HW_6_AI_agent\.env
API key loaded successfully


## Крок 2. Створюємо текст із фінансовими даними



In [3]:
financial_text = """ТехНова Інк за 2024 рік: виручка 450 млн дол (+18 відсотків), чистий прибуток 67 млн дол, маржа EBITDA 22 відсотки.
Працівників 2300 осіб. ROE становить 15.2 відсотка. Співвідношення борг до EBITDA дорівнює 1.8. Штаб-квартира у Львові.

ГрінЕнерджі Солюшнс за 2024 рік: виручка 890 млн дол (+35 відсотків), чистий прибуток 156 млн дол, маржа EBITDA 28 відсотків.
Працівників 5100 осіб. ROE становить 19.7 відсотка. Співвідношення борг до EBITDA дорівнює 0.9. Лідер у секторі відновлюваної енергії.

РітейлМакс Груп за 2024 рік: виручка 1200 млн дол (+8 відсотків), чистий прибуток 48 млн дол, маржа EBITDA 12 відсотків.
Працівників 8500 осіб. ROE становить 9.3 відсотка. Співвідношення борг до EBITDA дорівнює 3.2. Мережа з 340 магазинів в Україні.

ФінТех Дайнамікс за 2024 рік: виручка 230 млн дол (+52 відсотки), чистий збиток мінус 15 млн дол, маржа EBITDA мінус 8 відсотків.
Працівників 890 осіб. Від'ємний показник ROE. Стартап який активно інвестує у масштабування бізнесу."""

print(financial_text)

ТехНова Інк за 2024 рік: виручка 450 млн дол (+18 відсотків), чистий прибуток 67 млн дол, маржа EBITDA 22 відсотки.
Працівників 2300 осіб. ROE становить 15.2 відсотка. Співвідношення борг до EBITDA дорівнює 1.8. Штаб-квартира у Львові.

ГрінЕнерджі Солюшнс за 2024 рік: виручка 890 млн дол (+35 відсотків), чистий прибуток 156 млн дол, маржа EBITDA 28 відсотків.
Працівників 5100 осіб. ROE становить 19.7 відсотка. Співвідношення борг до EBITDA дорівнює 0.9. Лідер у секторі відновлюваної енергії.

РітейлМакс Груп за 2024 рік: виручка 1200 млн дол (+8 відсотків), чистий прибуток 48 млн дол, маржа EBITDA 12 відсотків.
Працівників 8500 осіб. ROE становить 9.3 відсотка. Співвідношення борг до EBITDA дорівнює 3.2. Мережа з 340 магазинів в Україні.

ФінТех Дайнамікс за 2024 рік: виручка 230 млн дол (+52 відсотки), чистий збиток мінус 15 млн дол, маржа EBITDA мінус 8 відсотків.
Працівників 890 осіб. Від'ємний показник ROE. Стартап який активно інвестує у масштабування бізнесу.


## Крок 3. Розбиваємо текст на чанки через RecursiveCharacterTextSplitter




In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n"],
    chunk_size=300,
    chunk_overlap=0,
)

documents = text_splitter.create_documents([financial_text.strip()])

print(f"Кількість чанків: {len(documents)}")

Кількість чанків: 4


Тепер подивимось на кожен чанк окремо.

Це навчальна перевірка: ми хочемо очима побачити, що кожен документ містить дані тільки однієї компанії.

Якщо тут 4 чанки і кожен починається з окремої компанії, критерій чанкування виконано.

In [5]:
for i, document in enumerate(documents, 1):
    print("=" * 70)
    print(f"Чанк {i}")
    print("=" * 70)
    print(document.page_content)
    print()

Чанк 1
ТехНова Інк за 2024 рік: виручка 450 млн дол (+18 відсотків), чистий прибуток 67 млн дол, маржа EBITDA 22 відсотки.
Працівників 2300 осіб. ROE становить 15.2 відсотка. Співвідношення борг до EBITDA дорівнює 1.8. Штаб-квартира у Львові.

Чанк 2
ГрінЕнерджі Солюшнс за 2024 рік: виручка 890 млн дол (+35 відсотків), чистий прибуток 156 млн дол, маржа EBITDA 28 відсотків.
Працівників 5100 осіб. ROE становить 19.7 відсотка. Співвідношення борг до EBITDA дорівнює 0.9. Лідер у секторі відновлюваної енергії.

Чанк 3
РітейлМакс Груп за 2024 рік: виручка 1200 млн дол (+8 відсотків), чистий прибуток 48 млн дол, маржа EBITDA 12 відсотків.
Працівників 8500 осіб. ROE становить 9.3 відсотка. Співвідношення борг до EBITDA дорівнює 3.2. Мережа з 340 магазинів в Україні.

Чанк 4
ФінТех Дайнамікс за 2024 рік: виручка 230 млн дол (+52 відсотки), чистий збиток мінус 15 млн дол, маржа EBITDA мінус 8 відсотків.
Працівників 890 осіб. Від'ємний показник ROE. Стартап який активно інвестує у масштабування 

In [6]:
assert len(documents) == 4, f"Очікували 4 чанки, але отримали {len(documents)}"

print("Перевірка пройдена: кожна компанія є окремим чанком.")

Перевірка пройдена: кожна компанія є окремим чанком.


## Крок 4. Створюємо embeddings і Chroma vector store




In [7]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

collection_name = 'finansovi_zvity_ukrainskyh_kompanii'

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name=collection_name,
)

print(f'Створено Chroma collection: {collection_name}')
print(f'Документів у vector store: {vector_store._collection.count()}')


Створено Chroma collection: finansovi_zvity_ukrainskyh_kompanii
Документів у vector store: 4


## Крок 5. Налаштовуємо retriever



In [8]:
retriever = vector_store.as_retriever(search_kwargs={'k': 2})

test_question = 'Яка виручка у компанії ТехНова Інк за 2024 рік?'
relevant_documents = retriever.invoke(test_question)

print(f'Питання: {test_question}')
print(f'Знайдено документів: {len(relevant_documents)}')
print()

for i, document in enumerate(relevant_documents, 1):
    print('=' * 70)
    print(f'Документ {i}')
    print('=' * 70)
    print(document.page_content)
    print()


Питання: Яка виручка у компанії ТехНова Інк за 2024 рік?
Знайдено документів: 2

Документ 1
ТехНова Інк за 2024 рік: виручка 450 млн дол (+18 відсотків), чистий прибуток 67 млн дол, маржа EBITDA 22 відсотки.
Працівників 2300 осіб. ROE становить 15.2 відсотка. Співвідношення борг до EBITDA дорівнює 1.8. Штаб-квартира у Львові.

Документ 2
ФінТех Дайнамікс за 2024 рік: виручка 230 млн дол (+52 відсотки), чистий збиток мінус 15 млн дол, маржа EBITDA мінус 8 відсотків.
Працівників 890 осіб. Від'ємний показник ROE. Стартап який активно інвестує у масштабування бізнесу.



In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

def format_documents(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

prompt = ChatPromptTemplate.from_template('''
Ти фінансовий аналітик. Відповідай українською мовою.
Використовуй тільки інформацію з контексту.
Якщо відповіді немає в контексті, напиши, що інформація відсутня.
Відповідай коротко, але з конкретними цифрами, якщо вони є.

Контекст:
{context}

Питання:
{question}
''')

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

rag_chain = (
    {
        'context': retriever | format_documents,
        'question': RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

test_questions = [
    'Яка виручка у компанії ТехНова Інк за 2024 рік?',
    'Скільки працівників у ГрінЕнерджі Солюшнс?',
    'Який показник ROE у РітейлМакс Груп?',
    'Де знаходиться штаб-квартира ТехНова Інк?',
    'Яка маржа EBITDA у ФінТех Дайнамікс?',
    'Який чистий прибуток у ЕкоПакування Солюшнс?',
]

for i, question in enumerate(test_questions, 1):
    answer = rag_chain.invoke(question)

    print('=' * 70)
    print(f'Питання {i}: {question}')
    print(f'Відповідь: {answer}')
    print('-' * 70)


Питання 1: Яка виручка у компанії ТехНова Інк за 2024 рік?
Відповідь: Виручка у компанії ТехНова Інк за 2024 рік становить 450 млн дол.
----------------------------------------------------------------------
Питання 2: Скільки працівників у ГрінЕнерджі Солюшнс?
Відповідь: У ГрінЕнерджі Солюшнс працює 5100 осіб.
----------------------------------------------------------------------
